In [1]:
import json
import time
from pathlib import Path

import pandas as pd
import requests
import yaml

# Project paths (the notebook lives in notebooks/)
ROOT = Path("..")
RAW_OSM = ROOT / "data" / "raw" / "osm"
INTERIM = ROOT / "data" / "interim"
RAW_OSM.mkdir(parents=True, exist_ok=True)
INTERIM.mkdir(parents=True, exist_ok=True)

with open(ROOT / "config.yaml") as f:
    config = yaml.safe_load(f)

COUNTRIES = config["scope"]["countries"]
PAUSE_S = config["osm"]["request_pause_s"]
print(len(COUNTRIES), "countries | pause:", PAUSE_S, "s")

ParserError: while parsing a block mapping
  in "../config.yaml", line 6, column 2
expected <block end>, but found '<block mapping start>'
  in "../config.yaml", line 10, column 3

In [ ]:
OVERPASS_URL = "https://overpass-api.de/api/interpreter"

def build_query(country_code: str) -> str:
    """Overpass QL query for data centres in one country (existing, under construction, planned)."""
    return f"""
    [out:json][timeout:180];
    area["ISO3166-1"="{country_code}"][admin_level=2]->.country;
    (
      nwr["telecom"="data_center"](area.country);
      nwr["building"="data_center"](area.country);
      nwr["industrial"="data_centre"](area.country);
      nwr["construction:telecom"="data_center"](area.country);
      nwr["construction"="data_center"](area.country);
      nwr["proposed:telecom"="data_center"](area.country);
    );
    out geom;
    """

print(build_query("PT"))

In [ ]:
HEADERS = {"User-Agent": "ia-agua-datacenters (portfolio project; github.com/Carol-Sust/Sustainability)"}

def fetch_country(country_code: str, force: bool = False, max_retries: int = 4) -> dict:
    """Download OSM data centres for one country, caching the raw JSON in data/raw/osm/."""
    cache_file = RAW_OSM / f"{country_code}.json"
    if cache_file.exists() and not force:
        return json.loads(cache_file.read_text())

    query = build_query(country_code)
    for attempt in range(1, max_retries + 1):
        response = requests.post(OVERPASS_URL, data={"data": query}, headers=HEADERS, timeout=300)
        if response.status_code == 200:
            data = response.json()
            cache_file.write_text(json.dumps(data))
            return data
        # 429 = too many requests, 504 = server busy -> wait and retry
        wait = 30 * attempt
        print(f"{country_code}: HTTP {response.status_code}, retry {attempt}/{max_retries} in {wait}s")
        time.sleep(wait)

    raise RuntimeError(f"{country_code}: failed after {max_retries} attempts")

In [ ]:
pt = fetch_country("PT")
elements = pt["elements"]
print("Elements found:", len(elements))

# How many points (node), polygons (way) and relations?
print(pd.Series([e["type"] for e in elements]).value_counts())

# Tags of each element as a table
tags = pd.json_normalize([e.get("tags", {}) for e in elements])
cols = [c for c in ["name", "operator", "telecom", "building", "construction", "proposed:telecom"] if c in tags.columns]
tags[cols].head(20)

In [ ]:
from tqdm import tqdm

counts = {}
failed = []

for cc in tqdm(COUNTRIES):
    already_cached = (RAW_OSM / f"{cc}.json").exists()
    try:
        data = fetch_country(cc)
        counts[cc] = len(data["elements"])
    except RuntimeError as error:
        print(error)
        failed.append(cc)
    # Only pause after a real request, not when reading from cache
    if not already_cached:
        time.sleep(PAUSE_S)

print("Failed:", failed if failed else "none")

In [ ]:
!git add -A
!git commit -m "Step 1: Overpass download for all countries"
!git push

In [ ]:
saved = sorted(p.stem for p in RAW_OSM.glob("*.json"))
missing = sorted(set(COUNTRIES) - set(saved))
print(len(saved), "files | missing:", missing if missing else "none")

In [ ]:
counts = {cc: len(json.loads((RAW_OSM / f"{cc}.json").read_text())["elements"]) for cc in saved}
summary = pd.Series(counts, name="elements").sort_values(ascending=False)
print("Total elements:", summary.sum())
summary